Parse GO enrichment CSVs and summarize

In [1]:

import pandas as pd

# Load both enrichment files
up = pd.read_csv('/mnt/user-uploads/GO_enrichment_OSD498_510_radiation_upregulated.csv')
down = pd.read_csv('/mnt/user-uploads/GO_enrichment_OSD498_510_radiation_downregulated.csv')

print("=== UPREGULATED ===")
print(f"Total GO terms: {len(up)}")
print(f"Columns: {list(up.columns)}")
print(f"\nTop 30 by p.adjust:")
print(up[['ID','Description','Count','FoldEnrichment','p.adjust']].head(30).to_string())

print("\n\n=== DOWNREGULATED ===")
print(f"Total GO terms: {len(down)}")
print(f"\nTop 30 by p.adjust:")
print(down[['ID','Description','Count','FoldEnrichment','p.adjust']].head(30).to_string())

# Count unique genes
up_genes = set()
for gids in up['geneID']:
    up_genes.update(gids.split('/'))
down_genes = set()
for gids in down['geneID']:
    down_genes.update(gids.split('/'))

print(f"\n\nUnique genes in upregulated pathways: {len(up_genes)}")
print(f"Unique genes in downregulated pathways: {len(down_genes)}")
print(f"Overlap: {len(up_genes & down_genes)}")


=== UPREGULATED ===
Total GO terms: 226
Columns: ['ID', 'Description', 'GeneRatio', 'BgRatio', 'RichFactor', 'FoldEnrichment', 'zScore', 'pvalue', 'p.adjust', 'qvalue', 'geneID', 'Count']

Top 30 by p.adjust:
            ID                                             Description  Count  FoldEnrichment      p.adjust
0   GO:0042742                           defense response to bacterium    193        2.826490  7.555031e-44
1   GO:0071456                            cellular response to hypoxia    133        3.494562  1.156664e-42
2   GO:0036294            cellular response to decreased oxygen levels    133        3.465441  2.097506e-42
3   GO:0071453                      cellular response to oxygen levels    133        3.465441  2.097506e-42
4   GO:0001666                                     response to hypoxia    141        3.290049  1.392519e-41
5   GO:0036293                     response to decreased oxygen levels    141        3.241666  1.136996e-40
6   GO:0070482                     

Run PDF generation script

In [3]:

import subprocess
result = subprocess.run(["python", "/workspace/generate_strategy_pdf.py"], capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)


STDOUT: 
STDERR: Traceback (most recent call last):
  File "/workspace/generate_strategy_pdf.py", line 12, in <module>
    from reportlab.lib.units import inch, pt
ImportError: cannot import name 'pt' from 'reportlab.lib.units' (/opt/conda/lib/python3.11/site-packages/reportlab/lib/units.py)

Return code: 1


Run PDF generation script (retry)

In [5]:

import subprocess
result = subprocess.run(["python", "/workspace/generate_strategy_pdf.py"], capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)


STDOUT: 
STDERR: Traceback (most recent call last):
  File "/workspace/generate_strategy_pdf.py", line 66, in <module>
    styles.add(ParagraphStyle(name="Bullet", fontName="Helvetica",
  File "/opt/conda/lib/python3.11/site-packages/reportlab/lib/styles.py", line 253, in add
    raise KeyError("Style '%s' already defined in stylesheet" % key)
KeyError: "Style 'Bullet' already defined in stylesheet"

Return code: 1


Run PDF generation script (retry 2)

In [7]:

import subprocess
result = subprocess.run(["python", "/workspace/generate_strategy_pdf.py"], capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)


STDOUT: PDF generated: /mnt/results/report_radiation_response_strategy.pdf
File size: 1051887 bytes

STDERR: 
Return code: 0


Validate PDF structure and content

In [9]:

from pypdf import PdfReader
import os

output_path = "/mnt/results/report_radiation_response_strategy.pdf"
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)

print(f"Page count: {page_count}")
print(f"File size: {file_size} bytes ({file_size/1024:.1f} KB)")

# Check text extractability on each page
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    text_len = len(text.strip())
    print(f"  Page {i+1}: {text_len} chars extractable")
    if text_len < 50:
        print(f"    WARNING: Page {i+1} may be blank or image-only")

# Sanity checks
assert page_count >= 5, f"Report has only {page_count} pages — likely missing content"
assert file_size > 5000, f"Report is only {file_size} bytes — likely blank or corrupt"
first_page_text = reader.pages[0].extract_text()
assert len(first_page_text.strip()) > 0, "First page has no extractable text"
print("\nAll validation checks passed.")


Page count: 18
File size: 1051887 bytes (1027.2 KB)
  Page 1: 2490 chars extractable
  Page 2: 67 chars extractable
  Page 3: 3725 chars extractable
  Page 4: 3412 chars extractable
  Page 5: 2906 chars extractable
  Page 6: 1416 chars extractable
  Page 7: 2469 chars extractable
  Page 8: 2496 chars extractable
  Page 9: 2704 chars extractable
  Page 10: 2325 chars extractable
  Page 11: 2738 chars extractable
  Page 12: 2761 chars extractable
  Page 13: 103 chars extractable
  Page 14: 2714 chars extractable
  Page 15: 1189 chars extractable
  Page 16: 2365 chars extractable
  Page 17: 911 chars extractable
  Page 18: 3661 chars extractable

All validation checks passed.


Check low-text pages for issues

In [11]:

from pypdf import PdfReader

reader = PdfReader("/mnt/results/report_radiation_response_strategy.pdf")

# Check the low-text pages
for i in [1, 12]:  # 0-indexed: pages 2 and 13
    text = reader.pages[i].extract_text()
    print(f"=== Page {i+1} text ({len(text.strip())} chars) ===")
    print(repr(text.strip()[:500]))
    print()


=== Page 2 text (67 chars) ===
'Targeted Analytical Strategy: Arabidopsis Radiation Response\nPage 2'

=== Page 13 text (103 chars) ===
'Targeted Analytical Strategy: Arabidopsis Radiation Response\nPage 13\nplant biology journals prioritize.'



Regenerate PDF with layout fix

In [13]:

import subprocess
result = subprocess.run(["python", "/workspace/generate_strategy_pdf.py"], capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)


STDOUT: PDF generated: /mnt/results/report_radiation_response_strategy.pdf
File size: 1051242 bytes

STDERR: 
Return code: 0


Re-validate PDF after layout fix

In [15]:

from pypdf import PdfReader
import os

output_path = "/mnt/results/report_radiation_response_strategy.pdf"
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)

print(f"Page count: {page_count}")
print(f"File size: {file_size/1024:.1f} KB")

blank_pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    text_len = len(text.strip())
    status = "OK" if text_len > 100 else "CHECK"
    if text_len < 100:
        blank_pages.append(i+1)
    print(f"  Page {i+1}: {text_len} chars  [{status}]")

if blank_pages:
    print(f"\nPotential blank/low-text pages: {blank_pages}")
else:
    print("\nNo blank pages detected.")

# Final assertions
assert page_count >= 5
assert file_size > 5000
assert len(blank_pages) == 0, f"Blank pages remain: {blank_pages}"
print("All validation checks passed.")


Page count: 17
File size: 1026.6 KB
  Page 1: 2490 chars  [OK]
  Page 2: 3725 chars  [OK]
  Page 3: 3412 chars  [OK]
  Page 4: 2906 chars  [OK]
  Page 5: 1416 chars  [OK]
  Page 6: 2469 chars  [OK]
  Page 7: 2496 chars  [OK]
  Page 8: 2704 chars  [OK]
  Page 9: 2324 chars  [OK]
  Page 10: 2738 chars  [OK]
  Page 11: 2761 chars  [OK]
  Page 12: 103 chars  [OK]
  Page 13: 2714 chars  [OK]
  Page 14: 1189 chars  [OK]
  Page 15: 2365 chars  [OK]
  Page 16: 911 chars  [OK]
  Page 17: 3661 chars  [OK]

No blank pages detected.
All validation checks passed.


Regenerate PDF with orphan fix

In [17]:

import subprocess
result = subprocess.run(["python", "/workspace/generate_strategy_pdf.py"], capture_output=True, text=True, timeout=120)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)


STDOUT: PDF generated: /mnt/results/report_radiation_response_strategy.pdf
File size: 1050546 bytes

STDERR: 
Return code: 0


Final PDF validation

In [19]:

from pypdf import PdfReader
import os

output_path = "/mnt/results/report_radiation_response_strategy.pdf"
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)

print(f"Page count: {page_count}")
print(f"File size: {file_size/1024:.1f} KB")

low_text_pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    text_len = len(text.strip())
    if text_len < 100:
        low_text_pages.append((i+1, text_len))
    print(f"  Page {i+1}: {text_len} chars")

if low_text_pages:
    print(f"\nLow-text pages: {low_text_pages}")
else:
    print("\nAll pages have substantial content. No blank or orphan pages.")

assert page_count >= 5
assert file_size > 5000
print("Validation passed.")


Page count: 16
File size: 1025.9 KB
  Page 1: 2490 chars
  Page 2: 3725 chars
  Page 3: 3412 chars
  Page 4: 2906 chars
  Page 5: 1416 chars
  Page 6: 2469 chars
  Page 7: 2496 chars
  Page 8: 2704 chars
  Page 9: 2324 chars
  Page 10: 2738 chars
  Page 11: 2761 chars
  Page 12: 2586 chars
  Page 13: 1352 chars
  Page 14: 2365 chars
  Page 15: 911 chars
  Page 16: 3661 chars

All pages have substantial content. No blank or orphan pages.
Validation passed.
